# 20_01 — Extract ResNet50 Embeddings (Fixed Encoder, Cached)

## Goal
Extract **2048-dim embeddings** from a pretrained **ResNet50 (ImageNet)** encoder (classifier head removed) for the fixed split **split_v1** and cache them to disk for reuse in downstream classical ML baselines.

This notebook is designed to be:
- **Re-runnable**: auto-detects cached embeddings and reuses them
- **Deterministic**: uses Phase 1 **evaluation transforms** (no randomness)
- **Efficient**: batches inference, uses GPU if available
- **Pipeline-friendly**: outputs `.npy` embeddings + labels for train/val/test

## Data Contract (Phase 1)
- Splits:
  - `data/splits/split_v1/train.csv`
  - `data/splits/split_v1/val.csv`
  - `data/splits/split_v1/test.csv`
- Class mapping:
  - `data/splits/split_v1/classes.json`
- Dataset images referenced by the CSV `filepath` column must exist on disk.

## Outputs (Cache)
Saved to:
- `data/processed/embeddings/split_v1/encoder_resnet50/`

Files:
- `train.npy`, `val.npy`, `test.npy`
- `labels_train.npy`, `labels_val.npy`, `labels_test.npy`
- `meta.json` (extraction metadata)

## MLflow
Logs parameters + metrics to project-level `mlruns/` and stores `meta.json` as an artifact.

In [ ]:
# Cell 1 — Imports

import os
import json
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet50_Weights

import mlflow

In [ ]:
# Cell 2 — Project root + paths (robust)

NOTEBOOK_DIR = Path.cwd().expanduser().resolve()

def find_project_root(start: Path) -> Path:
    markers_all = ["src", "configs"]
    markers_any = [".git", "pyproject.toml", "requirements.txt"]

    for p in [start] + list(start.parents):
        has_all = all((p / m).exists() for m in markers_all)
        has_any = any((p / m).exists() for m in markers_any)
        if has_all and has_any:
            return p

    # fallback for zipped/copy deployments without .git
    for p in [start] + list(start.parents):
        if all((p / m).exists() for m in markers_all):
            return p

    raise RuntimeError(f"Could not locate project root from: {start}")

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"

DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_ID = "split_v1"
SPLITS_DIR = DATA_DIR / "splits" / SPLIT_ID

TRAIN_CSV = SPLITS_DIR / "train.csv"
VAL_CSV = SPLITS_DIR / "val.csv"
TEST_CSV = SPLITS_DIR / "test.csv"
CLASSES_JSON = SPLITS_DIR / "classes.json"

ENCODER_NAME = "resnet50"
EMBEDDING_DIM = 2048
TRANSFORM_ID = "transforms_v1_eval_resize256_centercrop224_imagenetnorm"  # descriptive tag

EMB_DIR = DATA_DIR / "processed" / "embeddings" / SPLIT_ID / "encoder_resnet50"
EMB_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("EMB_DIR     :", EMB_DIR)

In [ ]:
# Cell 3 — MLflow setup

TRACKING_DIR = PROJECT_ROOT / "mlruns"
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

print("MLflow tracking URI:", TRACKING_DIR.as_uri())

In [ ]:
# Cell 4 — Cache check (skip extraction if embeddings already exist)

expected = {
    "train": EMB_DIR / "train.npy",
    "val": EMB_DIR / "val.npy",
    "test": EMB_DIR / "test.npy",
    "labels_train": EMB_DIR / "labels_train.npy",
    "labels_val": EMB_DIR / "labels_val.npy",
    "labels_test": EMB_DIR / "labels_test.npy",
    "meta": EMB_DIR / "meta.json",
}

CACHE_READY = all(p.exists() for p in expected.values())

if CACHE_READY:
    print("[CACHE] Embeddings already exist. Skipping extraction.")
    # quick shape peek
    X_train = np.load(expected["train"], mmap_mode="r")
    X_val = np.load(expected["val"], mmap_mode="r")
    X_test = np.load(expected["test"], mmap_mode="r")
    print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)
else:
    print("[CACHE MISS] Embeddings not found (or incomplete). Will extract.")

In [ ]:
# Cell 5 — Load splits + class mapping + normalize filepaths (portable across OS)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    class_to_idx = json.load(f)

idx_to_class = {v: k for k, v in class_to_idx.items()}

# Ensure label_idx exists
for df in (train_df, val_df, test_df):
    df["label_idx"] = df["label"].map(class_to_idx)
    assert df["label_idx"].isna().sum() == 0

def normalize_filepath(p: str) -> str:
    """
    Makes CSV filepaths portable across machines/OS.
    Handles:
      - Windows absolute paths containing /data/prepared/...
      - already-relative paths starting with data/...
      - backslashes -> slashes
    """
    p = str(p).strip()
    p2 = p.replace("\\", "/")

    marker = "/data/prepared/"
    low = p2.lower()
    if marker in low:
        idx = low.find(marker)
        rel = p2[idx+1:]  # "data/prepared/..."
        return str((PROJECT_ROOT / rel).resolve())

    if p2.startswith("data/"):
        return str((PROJECT_ROOT / p2).resolve())

    return p  # fallback

for df in (train_df, val_df, test_df):
    df["filepath"] = df["filepath"].apply(normalize_filepath)

print("Train/Val/Test sizes:", len(train_df), len(val_df), len(test_df))
print("Example path:", train_df["filepath"].iloc[0])
print("Exists?", Path(train_df["filepath"].iloc[0]).exists())

In [ ]:
# Cell 6 — Load Phase 1 eval transforms (deterministic)

# We reuse your Phase 1 transform pipeline from configs/transforms_v1.yaml
# This keeps embeddings consistent across runs.

try:
    from src.data.transforms import load_transforms_config, get_eval_transforms
except Exception as e:
    raise RuntimeError(
        "Could not import Phase 1 transforms from src.data.transforms.\n"
        "Make sure you're running from the project root and that src/ is on PYTHONPATH.\n"
        f"Original error: {e}"
    )

TRANSFORMS_CONFIG_PATH = CONFIGS_DIR / "transforms_v1.yaml"
tfm_cfg = load_transforms_config(str(TRANSFORMS_CONFIG_PATH))
eval_tfm = get_eval_transforms(tfm_cfg)

print("[OK] Loaded eval transforms from:", TRANSFORMS_CONFIG_PATH)

In [ ]:
# Cell 7 — Dataset + DataLoader (no shuffle; embedding order must match CSV order)

class CSVDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["filepath"].tolist()
        self.labels = df["label_idx"].astype(np.int64).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx: int):
        from PIL import Image  # local import avoids global dependency ordering issues
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        y = self.labels[idx]
        return img, y

# Device + loader params
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Reasonable defaults; adjust if needed
BATCH_SIZE = 128 if DEVICE == "cuda" else 64
NUM_WORKERS = min(8, os.cpu_count() or 2)
PIN_MEMORY = True if DEVICE == "cuda" else False

train_ds = CSVDataset(train_df, eval_tfm)
val_ds = CSVDataset(val_df, eval_tfm)
test_ds = CSVDataset(test_df, eval_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print("DEVICE:", DEVICE)
print("BATCH_SIZE:", BATCH_SIZE, "NUM_WORKERS:", NUM_WORKERS, "PIN_MEMORY:", PIN_MEMORY)

In [ ]:
# Cell 8 — Build ResNet50 encoder (feature extractor only)

# Use official torchvision weights for ImageNet
weights = ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

# Remove classifier head; output after avgpool -> flatten => 2048-dim
model.fc = torch.nn.Identity()

model.to(DEVICE)
model.eval()

# quick shape check
with torch.no_grad():
    x0, _ = next(iter(train_loader))
    x0 = x0.to(DEVICE)
    z0 = model(x0)
    assert z0.ndim == 2 and z0.shape[1] == EMBEDDING_DIM, f"Unexpected embedding shape: {z0.shape}"
    print("[OK] Encoder output shape:", tuple(z0.shape))

In [ ]:
# Cell 9 — Embedding extraction (single split)

def extract_embeddings(loader: DataLoader, encoder: torch.nn.Module, device: str) -> tuple[np.ndarray, np.ndarray]:
    embs = []
    ys = []

    t0 = time.time()
    n_seen = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            z = encoder(xb)  # (B, 2048)
            z = z.detach().float().cpu().numpy().astype(np.float32)

            embs.append(z)
            ys.append(yb.numpy().astype(np.int64))

            n_seen += z.shape[0]

    X = np.vstack(embs)
    y = np.concatenate(ys)

    dt = time.time() - t0
    ips = n_seen / dt if dt > 0 else float("inf")
    return X, y, dt, ips

In [ ]:
# Cell 10 — Run extraction + save atomically (only if cache missing)

def atomic_save_npy(path: Path, array: np.ndarray):
    tmp = path.with_suffix(path.suffix + ".tmp")
    np.save(tmp, array)
    tmp.replace(path)

def atomic_save_json(path: Path, obj: dict):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

if not CACHE_READY:
    print("[RUN] Extracting embeddings...")

    X_train, y_train, dt_train, ips_train = extract_embeddings(train_loader, model, DEVICE)
    X_val, y_val, dt_val, ips_val = extract_embeddings(val_loader, model, DEVICE)
    X_test, y_test, dt_test, ips_test = extract_embeddings(test_loader, model, DEVICE)

    print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

    # Save arrays atomically (prevents partial cache if interrupted)
    atomic_save_npy(expected["train"], X_train)
    atomic_save_npy(expected["val"], X_val)
    atomic_save_npy(expected["test"], X_test)
    atomic_save_npy(expected["labels_train"], y_train)
    atomic_save_npy(expected["labels_val"], y_val)
    atomic_save_npy(expected["labels_test"], y_test)

    meta = {
        "encoder_name": ENCODER_NAME,
        "embedding_dim": EMBEDDING_DIM,
        "split_id": SPLIT_ID,
        "transform_id": TRANSFORM_ID,
        "timestamp": datetime.now().isoformat(),
        "device": DEVICE,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "train_count": int(X_train.shape[0]),
        "val_count": int(X_val.shape[0]),
        "test_count": int(X_test.shape[0]),
        "timing_sec": {"train": dt_train, "val": dt_val, "test": dt_test},
        "throughput_img_per_sec": {"train": ips_train, "val": ips_val, "test": ips_test},
        "weights": str(weights),
    }
    atomic_save_json(expected["meta"], meta)

    print("[OK] Saved embeddings + labels + meta.json to:", EMB_DIR)
else:
    print("[SKIP] Cache is ready; nothing to extract.")

In [ ]:
# Cell 11 — Sanity checks (after extraction or cache reuse)

X_train = np.load(expected["train"], mmap_mode="r")
X_val = np.load(expected["val"], mmap_mode="r")
X_test = np.load(expected["test"], mmap_mode="r")

y_train = np.load(expected["labels_train"])
y_val = np.load(expected["labels_val"])
y_test = np.load(expected["labels_test"])

assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]
assert X_test.shape[0] == y_test.shape[0]

assert X_train.shape[1] == EMBEDDING_DIM
assert X_val.shape[1] == EMBEDDING_DIM
assert X_test.shape[1] == EMBEDDING_DIM

assert np.isfinite(np.asarray(X_train[:1000])).all()
assert set(np.unique(y_train)).issubset(set(class_to_idx.values()))

print("[OK] Embeddings sanity checks passed.")
print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

In [ ]:
# Cell 12 — MLflow logging (do NOT log large .npy artifacts)

meta_path = expected["meta"]
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

with mlflow.start_run(run_name=f"embeddings_{ENCODER_NAME}_{SPLIT_ID}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
    mlflow.log_param("stage", "embedding_extraction")
    mlflow.log_param("encoder_name", ENCODER_NAME)
    mlflow.log_param("embedding_dim", EMBEDDING_DIM)
    mlflow.log_param("split_id", SPLIT_ID)
    mlflow.log_param("transform_id", TRANSFORM_ID)
    mlflow.log_param("device", meta.get("device"))
    mlflow.log_param("batch_size", meta.get("batch_size"))
    mlflow.log_param("num_workers", meta.get("num_workers"))

    # Metrics: throughput (if available)
    tput = meta.get("throughput_img_per_sec", {})
    if "train" in tput:
        mlflow.log_metric("train_img_per_sec", float(tput["train"]))
    if "val" in tput:
        mlflow.log_metric("val_img_per_sec", float(tput["val"]))
    if "test" in tput:
        mlflow.log_metric("test_img_per_sec", float(tput["test"]))

    # Log meta.json as artifact (small + useful)
    mlflow.log_artifact(str(meta_path))

print("[OK] MLflow logging complete. (Embeddings not logged as artifacts to avoid bloat.)")